In [1]:
# 1. 구글 드라이브 마운트
from google.colab import drive
import os
import glob
import zipfile
import shutil

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
from google.colab import files

# 저장할 로컬 폴더 생성
train_label_dir = Path("/content/data/train/label")
val_label_dir = Path("/content/data/val/label")
train_label_dir.mkdir(parents=True, exist_ok=True)
val_label_dir.mkdir(parents=True, exist_ok=True)

# 구글 드라이브 내 zip 파일들 검색 (경로가 다르면 파일이 있는 폴더 경로로 맞춰주세요)
# 보통 내 드라이브에 바로가기 추가 시: '/content/drive/MyDrive/**/*.zip'
print("구글 드라이브에서 zip 파일 탐색 중...")
all_zips = sorted(glob.glob("/content/drive/MyDrive/**/대학부*.zip", recursive=True))

if not all_zips:
    # 혹시 이름이 다를 경우 드라이브 전체 zip 검색
    all_zips = sorted(glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True))

print(f"발견된 zip 파일 총 {len(all_zips)}개:")
for z in all_zips:
    print(" -", os.path.basename(z))

# 001~013 돌면서 JSON 파일만 추출
total_train_json = 0
total_val_json = 0

print("\nJSON 라벨 추출 시작 (음성 WAV는 건너뜁니다)...")
for zf_path in all_zips:
    zf_name = os.path.basename(zf_path)
    try:
        with zipfile.ZipFile(zf_path, 'r') as z:
            namelist = z.namelist()
            train_jsons = [n for n in namelist if 'training' in n.lower() and n.endswith('.json')]
            val_jsons = [n for n in namelist if 'validation' in n.lower() and n.endswith('.json')]

            # 파일 풀기 (폴더 구조 무시하고 파일명만 순수하게 넣기)
            for tj in train_jsons:
                filename = os.path.basename(tj)
                with z.open(tj) as src, open(train_label_dir / filename, "wb") as dst:
                    dst.write(src.read())

            for vj in val_jsons:
                filename = os.path.basename(vj)
                with z.open(vj) as src, open(val_label_dir / filename, "wb") as dst:
                    dst.write(src.read())

            total_train_json += len(train_jsons)
            total_val_json += len(val_jsons)
            print(f"[{zf_name}] 추출 완료 👉 Train JSON: {len(train_jsons)}개 / Val JSON: {len(val_jsons)}개")
    except Exception as e:
        print(f"[{zf_name}] 처리 중 에러: {e}")

print("\n" + "="*50)
print(f"전체 추출 완료!")
print(f"총 Training JSON: {total_train_json}개")
print(f"총 Validation JSON: {total_val_json}개")
print("="*50)

# 추출한 data 폴더를 압축
shutil.make_archive("/content/mission3_labels", 'zip', "/content/data")
print("mission3_labels.zip 생성 ")

# 내 컴퓨터로 즉시 다운로드 창 띄우기
files.download("/content/mission3_labels.zip")


🔍 구글 드라이브에서 zip 파일 탐색 중...
📦 발견된 zip 파일 총 13개:
 - 대학부 데이터-20260830T163301Z-1-001.zip
 - 대학부 데이터-20260830T163301Z-1-002.zip
 - 대학부 데이터-20260830T163301Z-1-003.zip
 - 대학부 데이터-20260830T163301Z-1-004.zip
 - 대학부 데이터-20260830T163301Z-1-005.zip
 - 대학부 데이터-20260830T163301Z-1-006.zip
 - 대학부 데이터-20260830T163301Z-1-007.zip
 - 대학부 데이터-20260830T163301Z-1-008.zip
 - 대학부 데이터-20260830T163301Z-1-009.zip
 - 대학부 데이터-20260830T163301Z-1-010.zip
 - 대학부 데이터-20260830T163301Z-1-011.zip
 - 대학부 데이터-20260830T163301Z-1-012.zip
 - 대학부 데이터-20260830T163301Z-1-013.zip

🚀 JSON 라벨 추출 시작 (음성 WAV는 건너뜁니다)...
[대학부 데이터-20260830T163301Z-1-001.zip] 추출 완료 👉 Train JSON: 4060개 / Val JSON: 2055개
[대학부 데이터-20260830T163301Z-1-002.zip] 추출 완료 👉 Train JSON: 4192개 / Val JSON: 993개
[대학부 데이터-20260830T163301Z-1-003.zip] 추출 완료 👉 Train JSON: 2210개 / Val JSON: 491개
[대학부 데이터-20260830T163301Z-1-004.zip] 추출 완료 👉 Train JSON: 3393개 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>